In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    silhouette_score
)


In [2]:

# Load datasets
train = pd.read_csv("../training_image_features_grouped.csv")
test = pd.read_csv("../testing_image_features_grouped.csv")

TARGET = "Condition_Class"
CLASSES = ["Excellent", "Good", "Fair", "Poor"]

print("Training Dataset Shape:", train.shape)
print("Testing Dataset Shape:", test.shape)

print("\nTraining Dataset:")
display(train.head())


Training Dataset Shape: (11465, 43)
Testing Dataset Shape: (2644, 43)

Training Dataset:


,Sample_ID,Image_ID,Material_Type,Material_Subtype,Source,Exposure_Type,Crack_Level,Surface_Damage,Breakage_Level,Discoloration_Level,...,dark_pixel_ratio,bright_pixel_ratio,mean_hue,mean_saturation,saturation_std,contour_area,contour_perimeter,circularity,solidity,extent
0,TRAINING_00013,10.jpg,brick,Standard,Demolition_Site,Exterior_Weathered,5,3.5,2,Moderate,...,0.013321,0.141387,16.121604,106.880770,22.460271,16131.0,745.470123,0.364763,0.928938,0.768143
1,TRAINING_00014,10.jpg,concrete,Reinforced,Demolition_Site,Exterior_Weathered,5,4.0,3,High,...,0.017225,0.326960,19.273151,94.502765,23.173802,6688.0,441.102593,0.431943,0.901408,0.557891
2,TRAINING_00015,10.jpg,tile,Standard,Demolition_Site,Exterior_Weathered,3,3.5,4,High,...,0.118015,0.170967,29.613107,51.448496,24.297176,10975.5,720.884337,0.265401,0.810628,0.683917
3,TRAINING_00016,10.jpg,brick,Standard,Demolition_Site,Exterior_Weathered,5,3.5,3,Moderate,...,0.009063,0.008142,13.644708,119.697837,26.109004,19055.5,769.251868,0.404663,0.893785,0.598101
4,TRAINING_00017,10.jpg,concrete,Reinforced,Demolition_Site,Exterior_Weathered,5,3.5,3,Moderate,...,0.000417,0.306625,19.245853,95.316954,25.873127,7753.5,720.364566,0.187760,0.805977,0.488502


In [3]:

# Exclude target-related scores
exclude = {
    "Condition_Score",
    "Damage_Score",
    "Alternative_Use_Suitability"
}

# Select numeric image features
from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

print("Number of Features:", len(features))
print("\nSelected Features:")
print(features)


Number of Features: 26

Selected Features:
['Crack_Level', 'Surface_Damage', 'Breakage_Level', 'Deformation_Level', 'pixel_img_width', 'pixel_img_height', 'pixel_obj_width', 'pixel_obj_height', 'relative_area', 'edge_density', 'fill_ratio', 'color_std', 'mean_brightness', 'std_brightness', 'contrast', 'texture_laplacian_var', 'dark_pixel_ratio', 'bright_pixel_ratio', 'mean_hue', 'mean_saturation', 'saturation_std', 'contour_area', 'contour_perimeter', 'circularity', 'solidity', 'extent']


In [4]:

# Extract selected features
X_train = train[features].replace([np.inf, -np.inf], np.nan)
X_test = test[features].replace([np.inf, -np.inf], np.nan)

# Handle missing values
imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)
print("Preprocessing Completed Successfully")


Training Data Shape: (11465, 26)
Testing Data Shape: (2644, 26)
Preprocessing Completed Successfully


In [5]:

# Initialize Gaussian Mixture Model
model = GaussianMixture(
    n_components=4,
    covariance_type="diag",
    random_state=42,
    n_init=3,
    max_iter=200,
    reg_covar=1e-6
)

# Fit GMM on training data
model.fit(X_train)

# Predict training and testing clusters
train_clusters = model.predict(X_train)
test_clusters = model.predict(X_test)

print("GMM Clustering Completed")

print("\nCluster Distribution:")
print(pd.Series(train_clusters).value_counts().sort_index())


GMM Clustering Completed

Cluster Distribution:
0    3246
1    2662
2    3018
3    2539
Name: count, dtype: int64


In [6]:

# Map clusters to majority condition classes
cluster_mapping = {}

for cluster in range(4):
    labels = train.loc[
        train_clusters == cluster, TARGET
    ]

    if len(labels) > 0:
        cluster_mapping[cluster] = labels.mode().iloc[0]
    else:
        cluster_mapping[cluster] = train[TARGET].mode().iloc[0]

print("Cluster to Condition Mapping:")
print(cluster_mapping)


Cluster to Condition Mapping:
{0: 'Fair', 1: 'Fair', 2: 'Fair', 3: 'Fair'}


In [7]:

# Predict testing classes
y_pred = np.array([
    cluster_mapping[c] for c in test_clusters
])

y_test = test[TARGET].astype(str)

print("Predicted Classes:")
print(pd.Series(y_pred).value_counts())

print("\nActual Classes:")
print(y_test.value_counts())


Predicted Classes:
Fair    2644
Name: count, dtype: int64

Actual Classes:
Condition_Class
Fair         1788
Good          586
Poor          238
Excellent      32
Name: count, dtype: int64


In [8]:

# Evaluate GMM
print("Gaussian Mixture Model")

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print("\nClassification Report:")

print(classification_report(
    y_test,
    y_pred,
    labels=CLASSES,
    zero_division=0
))

# Evaluate cluster separation
silhouette = silhouette_score(
    X_train,
    train_clusters,
    sample_size=2000,
    random_state=42
)

print("Silhouette Score:", silhouette)

print("BIC:", model.bic(X_train))
print("AIC:", model.aic(X_train))

print("Model Converged:", model.converged_)
print("Number of Iterations:", model.n_iter_)


Gaussian Mixture Model
Accuracy: 0.6762481089258698

Classification Report:
              precision    recall  f1-score   support

   Excellent       0.00      0.00      0.00        32
        Good       0.00      0.00      0.00       586
        Fair       0.68      1.00      0.81      1788
        Poor       0.00      0.00      0.00       238

    accuracy                           0.68      2644
   macro avg       0.17      0.25      0.20      2644
weighted avg       0.46      0.68      0.55      2644

Silhouette Score: 0.07779175497828318
BIC: 268877.0234715349
AIC: 267326.79503632826
Model Converged: True
Number of Iterations: 35


In [9]:
# Clustering is unsupervised: the accuracy above comes from mapping each cluster to its
# majority class, so compare it with the majority-class baseline and use label-free metrics.
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score, f1_score)
_y = test[TARGET].astype(str)
_X = X_train_pca if "X_train_pca" in globals() else X_train
print("Majority-class baseline accuracy:", round(_y.value_counts(normalize=True).max(), 4))
print("Macro F1 of cluster-to-class mapping:", round(f1_score(_y, y_pred, average="macro", zero_division=0), 4))
print("Adjusted Rand Index vs true labels (test):", round(adjusted_rand_score(_y, test_clusters), 4))
print("Normalized Mutual Information (test):", round(normalized_mutual_info_score(_y, test_clusters), 4))
print("Silhouette (train sample):", round(silhouette_score(_X, train_clusters, sample_size=2000, random_state=42), 4))


Majority-class baseline accuracy: 0.6762
Macro F1 of cluster-to-class mapping: 0.2017
Adjusted Rand Index vs true labels (test): 0.0376
Normalized Mutual Information (test): 0.0971
Silhouette (train sample): 0.0778


In [10]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

train_graph = pd.read_csv("../training_image_features_grouped.csv")

excluded = [
    "Condition_Class", "Condition_Score",
    "Damage_Score", "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X = train_graph[features].replace([np.inf, -np.inf], np.nan)
X = SimpleImputer(strategy="median").fit_transform(X)
X = StandardScaler().fit_transform(X)

sample_size = min(3000, len(X))
indices = np.random.RandomState(42).choice(
    len(X), sample_size, replace=False
)

model = GaussianMixture(
    n_components=4,
    covariance_type="diag",
    random_state=42,
    n_init=1,
    max_iter=100,
    reg_covar=1e-5
)

model.fit(X[indices])
clusters = model.predict(X[indices])

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X[indices])

plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=clusters, cmap="viridis",
    s=12, alpha=0.7
)

plt.title("Gaussian Mixture Model Cluster Visualization")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(scatter, label="GMM Cluster")
plt.tight_layout()
plt.show()

print("Samples visualized:", sample_size)


Samples visualized: 3000


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_29240\1560449145.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

train_graph = pd.read_csv("../training_image_features_grouped.csv")

excluded = [
    "Condition_Class", "Condition_Score",
    "Damage_Score", "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X = train_graph[features].replace([np.inf, -np.inf], np.nan)
X = SimpleImputer(strategy="median").fit_transform(X)
X = StandardScaler().fit_transform(X)

sample_size = min(3000, len(X))
indices = np.random.RandomState(42).choice(
    len(X), sample_size, replace=False
)

model = GaussianMixture(
    n_components=4,
    covariance_type="diag",
    random_state=42,
    n_init=1,
    max_iter=100,
    reg_covar=1e-5
)

model.fit(X[indices])
clusters = model.predict(X)

counts = np.bincount(clusters, minlength=4)

plt.figure(figsize=(8, 5))
plt.bar([f"Cluster {i}" for i in range(4)], counts)

plt.title("GMM: Number of Samples in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Samples")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_29240\3220515026.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

train_graph = pd.read_csv("../training_image_features_grouped.csv")

excluded = [
    "Condition_Class", "Condition_Score",
    "Damage_Score", "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X = train_graph[features].replace([np.inf, -np.inf], np.nan)
X = SimpleImputer(strategy="median").fit_transform(X)
X = StandardScaler().fit_transform(X)

sample_size = min(3000, len(X))
indices = np.random.RandomState(42).choice(
    len(X), sample_size, replace=False
)

model = GaussianMixture(
    n_components=4,
    covariance_type="diag",
    random_state=42,
    n_init=1,
    max_iter=100,
    reg_covar=1e-5
)

model.fit(X[indices])

probabilities = model.predict_proba(X[indices])
maximum_probabilities = probabilities.max(axis=1)

plt.figure(figsize=(9, 5))
plt.hist(maximum_probabilities, bins=20, edgecolor="black")

plt.title("GMM: Membership Probability Distribution")
plt.xlabel("Maximum Membership Probability")
plt.ylabel("Number of Samples")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_29240\1619055397.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

train_graph = pd.read_csv("../training_image_features_grouped.csv")

excluded = [
    "Condition_Class", "Condition_Score",
    "Damage_Score", "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X = train_graph[features].replace([np.inf, -np.inf], np.nan)
X = SimpleImputer(strategy="median").fit_transform(X)
X = StandardScaler().fit_transform(X)

sample_size = min(3000, len(X))
indices = np.random.RandomState(42).choice(
    len(X), sample_size, replace=False
)

model = GaussianMixture(
    n_components=4,
    covariance_type="diag",
    random_state=42,
    n_init=1,
    max_iter=100,
    reg_covar=1e-5
)

model.fit(X[indices])
clusters = model.predict(X)

distribution = pd.crosstab(
    pd.Series(clusters, name="Cluster"),
    train_graph["Condition_Class"].reset_index(drop=True)
)

distribution = distribution.reindex(
    columns=["Excellent", "Good", "Fair", "Poor"],
    fill_value=0
)

distribution.plot(kind="bar", figsize=(10, 6))

plt.title("GMM: Actual Condition Distribution in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0)
plt.legend(title="Condition")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_29240\1734074669.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

train_graph = pd.read_csv("../training_image_features_grouped.csv")
test_graph = pd.read_csv("../testing_image_features_grouped.csv")

target = "Condition_Class"
classes = ["Excellent", "Good", "Fair", "Poor"]

excluded = [
    target, "Condition_Score",
    "Damage_Score", "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X_train = train_graph[features].replace([np.inf, -np.inf], np.nan)
X_test = test_graph.reindex(columns=features).replace(
    [np.inf, -np.inf], np.nan
)

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

sample_size = min(3000, len(X_train))
indices = np.random.RandomState(42).choice(
    len(X_train), sample_size, replace=False
)

model = GaussianMixture(
    n_components=4,
    covariance_type="diag",
    random_state=42,
    n_init=1,
    max_iter=100,
    reg_covar=1e-5
)

model.fit(X_train[indices])

# Predict using the already trained GMM model
train_clusters = model.predict(X_train)
test_clusters = model.predict(X_test)

y_train = train_graph[target].reset_index(drop=True)

mapping = {}

for cluster in range(4):
    labels = y_train[train_clusters == cluster]

    mapping[cluster] = (
        labels.mode().iloc[0]
        if len(labels) > 0
        else y_train.mode().iloc[0]
    )

predictions = pd.Series(test_clusters).map(mapping)

cm = confusion_matrix(
    test_graph[target],
    predictions,
    labels=classes
)

fig, ax = plt.subplots(figsize=(8, 6))

ConfusionMatrixDisplay(
    cm,
    display_labels=classes
).plot(
    ax=ax,
    cmap="Blues",
    values_format="d"
)

plt.title("GMM Confusion Matrix")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

C:\Users\Ryan John\AppData\Local\Temp\ipykernel_29240\1931791779.py:92: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
